In [2]:
%pip install --upgrade git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-utroe0kv
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-utroe0kv
  Resolved https://github.com/huggingface/transformers.git to commit c9de1097eed992fe205f876415e7a7cfaa06be50
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 8.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 70.2 MB/s eta 0:00:00ta 0:00:01
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11875973 sha256=abf76cae33b93f83dc2b33f3aca04dd9b75692bf897d5d407bdc39dbbcc45d5c
  Stored in directory: /tmp/pip-ephem-wheel-cache-smo_h_sy/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: hf-xet

In [ ]:
import os
os._exit(0)

In [1]:
# load Gemma 4 model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
gemma_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [4]:
import numpy as np
import timm
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta
from shapely.geometry import Point
from geopy.distance import geodesic
import requests
import torch

class OilSpillDecisionTool:
    def __init__(self, coastline_path, ais_path):
        self.coastline_gdf = gpd.read_file(coastline_path)
        if self.coastline_gdf.crs is None:
            self.coastline_gdf = self.coastline_gdf.set_crs("EPSG:4326", allow_override=True)
        elif self.coastline_gdf.crs.to_string() != "EPSG:4326":
            self.coastline_gdf = self.coastline_gdf.to_crs("EPSG:4326")
        self.ais_df = pd.read_csv(ais_path)
        self.ais_df['timestamp'] = pd.to_datetime(self.ais_df['timestamp'])
        # ensure ’vessel_type‘ exist，if not fulfill default value
        if 'vessel_type' not in self.ais_df.columns:
            self.ais_df['vessel_type'] = 'unknown'

    def calculate_distance_to_coast(self, lon, lat):
        point = Point(lon, lat)
        coastline_union = self.coastline_gdf.geometry.union_all()
        nearest_point = coastline_union.interpolate(coastline_union.project(point))
        dist_km = geodesic((lat, lon), (nearest_point.y, nearest_point.x)).kilometers
        return dist_km / 1.852

    def find_recent_vessels(self, lon, lat, incident_time, hours=6, radius_nm=2):
        start_time = pd.to_datetime(incident_time) - timedelta(hours=hours)
        mask_time = (self.ais_df['timestamp'] >= start_time) & (self.ais_df['timestamp'] <= incident_time)
        df_time = self.ais_df[mask_time].copy()
        if df_time.empty:
            return []
        radius_km = radius_nm * 1.852
        # count distance
        distances = df_time.apply(lambda row: geodesic((row['latitude'], row['longitude']), (lat, lon)).km, axis=1)
        nearby = df_time[distances <= radius_km]
        # deduplicate
        vessels = nearby.drop_duplicates('mmsi')[['mmsi', 'vessel_type']].to_dict('records')
        return vessels

    def get_decision_chain(self, lon, lat, incident_time):
        distance_nm = self.calculate_distance_to_coast(lon, lat)
        vessels = self.find_recent_vessels(lon, lat, incident_time, hours=6, radius_nm=2)
        
        # wether is tanker
        has_tanker = any(v.get('vessel_type', '').lower() == 'tanker' for v in vessels)
        threshold = 50.0 if has_tanker else 12.0
        is_risk = distance_nm <= threshold
        
        return {
            "lon": lon,
            "lat": lat,
            "distance_nm": distance_nm,
            "vessels": vessels,
            "has_tanker": has_tanker,
            "threshold": threshold,
            "is_risk": is_risk,
        }

# ---------- init ----------
coastline_path = "/kaggle/input/datasets/qihuiren/oil-detect-dataset/kaggle_dataset/coastline/ne_10m_coastline.shp"
ais_path = "/kaggle/input/datasets/qihuiren/oil-detect-dataset/kaggle_dataset/ais/sample_ais.csv"
model_path = "/kaggle/input/datasets/qihuiren/oil-detect-dataset/kaggle_dataset/best_student.pth"

tool = OilSpillDecisionTool(coastline_path, ais_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classification_model = timm.create_model("mobilenetv3_small_100", num_classes=2, pretrained=False)
state_dict = torch.load(model_path, map_location=device)
classification_model.load_state_dict(state_dict)
classification_model = classification_model.to(device)
classification_model.eval()

print("✅ tools and model load successful")

✅ tools and model load successful


In [7]:
def generate_report_with_gemma(has_oil, decision_info, image_filename):
    lon = decision_info["lon"]
    lat = decision_info["lat"]
    dist = decision_info["distance_nm"]
    vessels = decision_info["vessels"]
    has_tanker = decision_info["has_tanker"]
    threshold = decision_info["threshold"]
    is_risk = decision_info["is_risk"]

    # no leak,just return
    if not has_oil:
        return f"No leak detected at coordinates [{lon:.4f}, {lat:.4f}]."
        
    coord_prefix = f"coordinates [{lon:.4f}, {lat:.4f}]"
    vessel_type_str = "tanker" if has_tanker else "non-tanker"
    
    if is_risk:
        # risk, find ship
        if vessels:
            vessel_ids = ", ".join(str(v['mmsi']) for v in vessels)
            data_desc = (f"leak at {coord_prefix}, vessel type {vessel_type_str}, "
                         f"{dist:.2f} nautical miles from coastline, less than {threshold} nautical miles, "
                         f"suspect vessels: [{vessel_ids}]")
            fallback_report = (f"There is a leak at {coord_prefix}, vessel type {vessel_type_str}, "
                               f"{dist:.2f} NM from coastline, less than {threshold} NM, suspect vessels: [{vessel_ids}]")
        else:
            # for robust
            data_desc = (f"leak at {coord_prefix}, vessel type {vessel_type_str}, "
                         f"{dist:.2f} NM from coastline, less than {threshold} NM, but no suspect vessels found")
            fallback_report = (f"There is a leak at {coord_prefix}, vessel type {vessel_type_str}, "
                               f"{dist:.2f} NM from coastline, less than {threshold} NM, but no suspect vessels found")
    else:
        # safe
        data_desc = (f"leak at {coord_prefix}, vessel type {vessel_type_str}, "
                     f"{dist:.2f} NM from coastline, greater than {threshold} NM, safe")
        fallback_report = (f"There is a leak at {coord_prefix}, vessel type {vessel_type_str}, "
                           f"{dist:.2f} NM from coastline, greater than {threshold} NM, safe")

    prompt = f"""You are a marine oil spill response assistant. Based on the following information, generate a short English sentence describing the situation. Do NOT include coordinates in your output (they will be added by the system).

Example:
Information: leak at coordinates [125.7, 36.2], vessel type tanker, 42.3 NM from coastline, less than 50 NM, suspect vessels: [816035178]
Output: There is a leak at coordinates [125.7, 36.2], vessel type tanker, 42.30 nautical miles from the coastline, less than 50 nautical miles, suspect vessels: [816035178]

Now information: {data_desc}
Output:"""

    # invoke Gemma
    inputs = tokenizer(prompt, return_tensors="pt").to(gemma_model.device)
    input_len = inputs['input_ids'].shape[1]
    outputs = gemma_model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
        repetition_penalty=1.1,
        top_p=0.9,
    )
    new_tokens = outputs[0][input_len:]
    gemma_desc = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # if output error，use fallback
    if not gemma_desc or "Example" in gemma_desc or gemma_desc.startswith("[") or "：" in gemma_desc[:10]:
        final_report = fallback_report
    else:
        # ensure output end by full stop，and no duplicate coordinates
        # Remove possible duplicate coordinate prefixes
        if gemma_desc.lower().startswith("there is a leak at coordinates"):
            final_report = gemma_desc
        else:
            # if not output as expected, concatenate coordinate prefixes
            final_report = f"There is a leak at {coord_prefix}. {gemma_desc}"
            # Clean up excess points
            final_report = final_report.replace("..", ".").strip()
    return final_report

In [9]:
# test process
from datetime import datetime
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

# Define validation set preprocessing (consistent with training)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_oil(image_path, model, transform, device, threshold=0.5):
    img = Image.open(image_path).convert('L')
    img = img.convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1)
        oil_prob = probs[0, 1].item()
    return oil_prob > threshold, oil_prob

# ---------- test data ----------
test_cases = [
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_dtsR0emPVPPNdxSS_SFr_cls_1.jpg",
        "lon": -89.382194,
        "lat": 28.801274,
        "timestamp": datetime(2024, 1, 9, 3, 0, 0),
        "filename": "test1.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_9c3d5585_TRI_cls_1.jpg",
        "lon": -89.814811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test2.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/0_0_0_img_5kPtgTDfaBqbQJtE_ADR_cls_0.jpg",
        "lon": -92.814811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test3.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_9c3d5585_TRI_cls_1.jpg",
        "lon": -92.814811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test4.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_9c3d5585_TRI_cls_1.jpg",
        "lon": -89.614811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test5.jpg"
    },
]

for case in test_cases:
    has_oil, prob = predict_oil(
        case["image_path"],
        classification_model,
        val_transform,
        device,
        threshold=0.5
    )

    if not has_oil:
        # no leak，directly generate a leak free report
        decision_info = tool.get_decision_chain(case["lon"], case["lat"], case["timestamp"])
        report = generate_report_with_gemma(has_oil, decision_info, case["filename"])
        print(f"=== report {case['filename']} ===\n{report}\n")
    else:
        # leak，obtain decision information
        decision_info = tool.get_decision_chain(case["lon"], case["lat"], case["timestamp"])
        report = generate_report_with_gemma(has_oil, decision_info, case["filename"])
        print(f"=== report {case['filename']} ===\n{report}\n")
        # save to file
        with open(f"{case['filename']}_report.txt", "w") as f:
            f.write(report)

=== report test1.jpg ===
There is a leak at coordinates [-89.3822, 28.8013], vessel type non-tanker, 7.78 nautical miles from coastline, less than 12.0 nautical miles, suspect vessels: [308457070]

=== report test2.jpg ===
There is a leak at coordinates [-89.8148, 29.2767], vessel type tanker, 2.30 nautical miles from the coastline, less than 50.0 nautical miles, suspect vessels: [816035178]

=== report test3.jpg ===
No leak detected at coordinates [-92.8148, 29.2767].

=== report test4.jpg ===
There is a leak at coordinates [-92.8148, 29.2767], vessel type non-tanker, 20.83 nautical miles from coastline, greater than 12.0 nautical miles, safe

=== report test5.jpg ===
There is a leak at coordinates [-89.6148, 29.2767], vessel type non-tanker, 0.24 nautical miles from the coastline, less than 12.0 nautical miles, but no suspect vessels found

